# 13 — Modelos de Linguagem e Geração de Texto

**Disciplina 06 — Processamento de Linguagem Natural**  |  **Capítulo 13/17**

> Biblioteca pessoal de Ciência de Dados · capítulo didático independente · núcleo executável sem internet


### Materiais complementares

- [Guia técnico EP — Modelos de Linguagem e Geração](./reference_materials/13_modelos_linguagem_geracao.pdf)
- [Cheat sheet cookbook — Modelos de Linguagem e Geração](./cheat_sheets/cheat_sheet_modelos_linguagem_geracao.pdf)
- [Documentação e saídas](./README.pt-BR.md)


## Objetivos de aprendizagem

- Entender previsão do próximo token e temperatura.
- Construir gerador probabilístico elementar e reproduzível.
- Distinguir geração linguística de veracidade factual.

## Rastreabilidade e pré-requisitos

- **Origem e referência:** Ampliação moderna; ponte com Deep Learning.
- **Conhecimentos reutilizados:** 03 — probabilidade; 05 — modelos.
- **Progressão:** novos conceitos são aprofundados aqui sem reconstruir capítulos anteriores.

## Fundamentos conceituais

Um modelo de linguagem autoregressivo estima a distribuição do próximo token condicionada ao prefixo. Temperatura altera a concentração da distribuição de amostragem: temperaturas baixas privilegiam tokens mais prováveis, sem tornar o texto necessariamente verdadeiro. Contexto máximo, tokenização, amostragem e instruções influenciam a geração. Grandes modelos possuem treinamento e alinhamento adicionais que não são reproduzidos neste laboratório.

## Modelo matemático e interpretação

P(w₁,…,wₙ)=∏ᵢP(wᵢ | w₁,…,wᵢ₋₁). Na amostragem, pᵢ(T)=softmax(logitsᵢ/T), para T>0.

## Estudo guiado

**Pergunta de investigação:** como demonstrar o conceito central com um exemplo pequeno, auditável e reproduzível?

Nas células seguintes, inspecione as entradas, execute os procedimentos e explique com suas próprias palavras o resultado. Ao trocar o texto ou o parâmetro, observe se o comportamento permanece coerente. Esses resultados são ilustrações pedagógicas, não benchmarks.

## Implementação reproduzível

O código abaixo usa textos sintéticos ou bibliotecas locais. **Não executa downloads nem acessa APIs externas.** Exemplos opcionais só rodam quando a dependência já está disponível; a ausência deles não interrompe o notebook.

In [1]:
from collections import defaultdict,Counter
import numpy as np
frases=['dados ajudam decisões','dados permitem análise','modelos geram respostas','modelos ajudam análise','análise melhora decisões']
trans=defaultdict(Counter)
for s in frases:
    tokens=['<inicio>']+s.split()+['<fim>']
    for a,b in zip(tokens,tokens[1:]):trans[a][b]+=1
rng=np.random.default_rng(42)
def amostrar(proximo,temperatura=1.):
    opcoes=list(proximo); cont=np.array([proximo[x] for x in opcoes],float)
    prob=(cont**(1/temperatura));prob=prob/prob.sum()
    return rng.choice(opcoes,p=prob)
palavras=[];atual='<inicio>'
for _ in range(12):
    atual=amostrar(trans[atual],temperatura=.8)
    if atual=='<fim>':break
    palavras.append(atual)
print('Amostra de bigramas:', ' '.join(palavras))

Amostra de bigramas: modelos geram respostas


**Interpretação guiada:** verifique se o resultado corresponde à fórmula e à hipótese discutidas. Anote o que muda com a entrada e quais conclusões **não** podem ser generalizadas a partir deste exemplo.

In [2]:
import numpy as np
logits=np.array([2.,1.,0.])
for t in [0.5,1.,2.]:
    z=logits/t;p=np.exp(z-z.max());p/=p.sum()
    print('T=',t,'->',p.round(3))

T= 0.5 -> [0.867 0.117 0.016]
T= 1.0 -> [0.665 0.245 0.09 ]
T= 2.0 -> [0.506 0.307 0.186]


**Interpretação guiada:** verifique se o resultado corresponde à fórmula e à hipótese discutidas. Anote o que muda com a entrada e quais conclusões **não** podem ser generalizadas a partir deste exemplo.

## Exercícios orientados

1. **Compreensão:** explique qual informação o método preserva e qual descarta. Relacione a resposta com os objetivos do capítulo.
2. **Experimento:** altere uma frase e um parâmetro do código. Registre a diferença e uma hipótese explicativa.
3. **Avaliação crítica:** descreva uma situação em que a técnica geraria uma resposta inadequada. Proponha uma mitigação.
4. **Transferência:** conecte o conceito a uma disciplina anterior e identifique o que não precisa ser revisto.

**Resolução comentada (diretriz de resposta):** o método representa somente um recorte do fenômeno linguístico; mudanças de entrada alteram a saída segundo a regra estudada. Uma boa análise identifica perda de informação, contexto e risco de generalização. Para a resposta específica, justifique com os valores efetivamente obtidos na execução acima.

## Erros comuns, limitações e boas práticas

O gerador local demonstrativo é um modelo de bigramas, não um LLM. Não enviar dados privados a APIs externas sem autorização; serviços externos podem implicar custos e limites.

- Separe demonstração de produção: datasets sintéticos não demonstram generalização.
- Registre versões das dependências e parâmetros; não inclua credenciais no notebook.
- Consulte a documentação oficial antes de adaptar códigos históricos de 2021.

## Consulta rápida

| Item | Resumo |
|---|---|
| Conceito | Modelos de Linguagem e Geração de Texto |
| Fonte histórica | Ampliação moderna; ponte com Deep Learning |
| Dependências do exemplo | Python, NumPy / scikit-learn / NLTK / spaCy conforme as células |
| Critério de sucesso | Interpretar as saídas e explicar limites |
| Próximo passo | Aplicar a técnica a outro corpus de demonstração |

## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** decodificação e alucinação. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **decodificação e alucinação** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **decodificação e alucinação** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
